# Meet AlexNet: loading a network and classifying one image
**Background:** Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *NeurIPS, 25*.

AlexNet was trained on **ImageNet**: about 1.2 million photos, each labelled with one of **1,000 categories**. In 2012 it won the ImageNet competition by a large margin and started the deep learning boom in computer vision.

**What you will do**
1. Load AlexNet with its trained weights
2. Print its architecture and look at its layers
3. Prepare one image for the network
4. Classify the image and look at the top-5 predictions

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

In [ ]:
# Find the course data in Sciebo (works even if the shared folder was renamed)
matches = list(Path.home().glob("sciebo/*/Week2/"))
if not matches:
    raise FileNotFoundError(
        "Course data not found. Did you connect Sciebo when starting the server, "
        "and accept the shared CNU folder in Sciebo?"
    )
DATA = matches[0]
IMAGE_FILE = DATA / "example.jpg"                            # the image we classify today
print(f"Using data from {DATA}")

## 1. Load AlexNet
The **architecture** is the structure of the network: which layers there are and how they are connected. The **weights** are the numbers the network learned during training. Without trained weights, AlexNet would just guess randomly.

In [ ]:
# Load AlexNet with its trained weights (like ResMem(pretrained=True)).
# The first time, the weights (about 230 MB) are downloaded from the internet;
# after that they are stored on your server and load quickly.
weights = models.AlexNet_Weights.IMAGENET1K_V1   # the official ImageNet-trained version
model = models.alexnet(weights=weights)

# .eval() switches off training-only behaviour (dropout), so predictions are stable
model = model.to(device).eval()

# The names of the 1,000 categories, in the order of the network's outputs
categories = weights.meta["categories"]
print(f"AlexNet knows {len(categories)} categories")

## 2. The architecture

In [ ]:
print(model)

AlexNet has two parts:

- **`features`**: 5 **convolutional layers** (`Conv2d`). Each one slides small filters over the image and detects patterns: edges and colours in the first layer, more complex shapes and object parts in later layers. After each one comes a **`ReLU`**, which sets negative values to zero, and sometimes a **`MaxPool2d`**, which shrinks the image by keeping the strongest response in each small region.
- **`classifier`**: 3 **fully connected layers** (`Linear`). These combine all the features into one decision. The last layer has **1,000 outputs**, one for each category. (`Dropout` is only used during training.)

In [ ]:
# Look at a single layer: the first convolutional layer
first_layer = model.features[0]
print(first_layer)

`Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), ...)` means: the input has **3** channels (red, green, blue), the layer has **64** filters, each filter is **11 × 11** pixels, and it moves **4** pixels at a time.

**Try it:** print `model.features[3]` and `model.classifier[6]`. How many filters does the second convolutional layer have? How many inputs does the last layer get?

In [ ]:
# How many numbers (weights) did AlexNet learn?
n_weights = sum(p.numel() for p in model.parameters())
print(f"AlexNet has {n_weights:,} weights")

## 3. Prepare one image
AlexNet always takes a 224 × 224 pixel image. Every image is **resized**, **cropped** to the centre, and its colours are **normalised** the same way as during training.

In [ ]:
img = Image.open(IMAGE_FILE).convert("RGB")       # convert("RGB") ensures 3 colour channels

# The full preparation that goes into the network
preprocess = transforms.Compose([
    transforms.Resize(256),        # shorter side -> 256 pixels
    transforms.CenterCrop(224),    # cut out the central 224 x 224 square
    transforms.ToTensor(),         # turn the image into numbers (PyTorch's tensor format)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # same colour normalisation
                         std=[0.229, 0.224, 0.225]),   # as in ImageNet training
])

# The same steps without the number conversion, so we can SHOW what the network sees
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img); axes[0].set_title(f"original {img.size[0]} x {img.size[1]}")
axes[1].imshow(what_network_sees(img)); axes[1].set_title("network input 224 x 224")
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
# The network expects a BATCH of images, so we add a batch dimension of size 1
x = preprocess(img).unsqueeze(0).to(device)
print(f"Input shape: {tuple(x.shape)}   (images, colour channels, height, width)")

## 4. Classify the image
**Before running the next cell**, write down what you think AlexNet will answer.

In [ ]:
# We only predict, we don't train: no gradients needed
with torch.no_grad():
    outputs = model(x)

print(f"Output shape: {tuple(outputs.shape)}   (images, categories)")

# Softmax turns the 1,000 raw outputs into probabilities that sum to 1
probs = torch.softmax(outputs, dim=1)[0]
print(f"Sum of all probabilities: {probs.sum().item():.2f}")

In [ ]:
# The 5 most likely categories
top_probs, top_idx = probs.topk(5)
for p, i in zip(top_probs, top_idx):
    print(f"{categories[i]:30s} {p.item():.3f}")

In [ ]:
# The same as a picture
labels = [categories[i] for i in top_idx][::-1]     # reversed, so the top guess is on top
values = [p.item() for p in top_probs][::-1]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [1, 1.7]})
axes[0].imshow(what_network_sees(img)); axes[0].axis("off")
axes[1].barh(labels, values, color="tab:blue")
axes[1].set_xlim(0, 1); axes[1].set_xlabel("probability")
for y, p in enumerate(values):
    axes[1].text(p + 0.01, y, f"{p:.2f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

**Think about it:**
- Was AlexNet right? Was it more or less specific than your guess?
- How sure was it? How much probability is left for the other 995 categories?
- Do the 2nd–5th guesses make sense? Why might the network consider them?

**Try it:** upload your own image (drag it into the file browser on the left), set `IMAGE_FILE = "your_file.jpg"` in the setup cell, and run sections 3 and 4 again.